In [1]:
import torch
from torch import nn
from torchrl.envs import PettingZooWrapper
from env_v0 import MahjongGameEnv
from torchrl.envs import TransformedEnv
from torchrl.envs.transforms import ActionMask
from torchrl.envs.utils import MarlGroupMapType
from torchrl.modules import MultiAgentConvNet, MultiAgentMLP, ProbabilisticActor, MaskedCategorical
from tensordict.nn import TensorDictModule
from torchrl.collectors import Collector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.objectives import ClipPPOLoss, ValueEstimators

env = PettingZooWrapper(env=MahjongGameEnv(), use_mask=True, return_state=True, categorical_actions=True, group_map=MarlGroupMapType.ALL_IN_ONE_GROUP)
print(env.reset())

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-21 14:07:52,024	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\torchrl\envs\libs\pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


TensorDict(
    fields={
        agents: TensorDict(
            fields={
                action_mask: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                mask: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: TensorDict(
                    fields={
                        observation: Tensor(shape=torch.Size([4, 156, 46]), device=cpu, dtype=torch.uint8, is_shared=False)},
                    batch_size=torch.Size([4]),
                    device=None,
                    is_shared=False),
                terminated: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([4]),
            device=None,
            is

In [76]:
device = 'cpu'

In [152]:
class CastToFloat(nn.Module):
    def forward(self, x):
        return x.float()   # or .to(torch.float32)

policy_net = nn.Sequential(
    nn.Flatten(-2),
    CastToFloat(),
    MultiAgentMLP(
        n_agent_inputs = 156 * 46,       
        n_agent_outputs = 75,       
        n_agents = 4,
        centralized = False,        
        share_params = True,      
        depth = 2,               
        num_cells = 1024,
        activation_class=torch.nn.Tanh
    )
)

policy_module = TensorDictModule(
    policy_net,
    in_keys=[("agents", "observation", "observation")],
    out_keys=[("agents", "logits")],
)

policy = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec_unbatched,
    in_keys={
        'logits': ('agents', 'logits'),
        'mask': ('agents', 'action_mask')
    }, # type: ignore
    out_keys=[env.action_key],
    distribution_class=MaskedCategorical,
    return_log_prob=True
)  # we'll need the log-prob for the PPO loss


In [153]:
critic_net = nn.Sequential(
    nn.Flatten(-2),                    # [248,46] -> [248*46]
    CastToFloat(),
    nn.Linear(248 * 46, 256),
    nn.Tanh(),
    # nn.Linear(256, 256),
    # nn.Tanh(),
    # nn.Linear(256, 256),
    # nn.Tanh(),
    # nn.Linear(256, 256),
    # nn.Tanh(),
    nn.Linear(256, 4),                # output 4 values (one per agent)
    nn.Unflatten(-1, (4, 1))          # reshape from [4] to [4,1]
)

critic = TensorDictModule(
    module=critic_net,
    in_keys=["state"],               # global state
    out_keys=[("agents", "state_value")],  # shape [4] values under agents
).to(device)

In [154]:
print("Running policy:", policy(env.reset()))
print("Running value:", critic(env.reset()))

Running policy: TensorDict(
    fields={
        agents: TensorDict(
            fields={
                action: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.int64, is_shared=False),
                action_log_prob: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.float32, is_shared=False),
                action_mask: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.bool, is_shared=False),
                done: Tensor(shape=torch.Size([4, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                logits: Tensor(shape=torch.Size([4, 75]), device=cpu, dtype=torch.float32, is_shared=False),
                mask: Tensor(shape=torch.Size([4]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: TensorDict(
                    fields={
                        observation: Tensor(shape=torch.Size([4, 156, 46]), device=cpu, dtype=torch.uint8, is_shared=False)},
                    batch_size=torch.Size([4]),
                    device

In [ ]:
# tensordict_data = None
# while True:
#     tensordict_data = env.rollout(1000)
#     if tensordict_data[('next', 'agents', 'reward')].any().item():
#         break
# print(tensordict_data[('next', 'agents', 'reward')])
# from pygame_visualizer import render_game_state
# import pygame
# import time
# pygame.init()
# screen = pygame.display.set_mode(size=(800, 800))
# font = pygame.font.Font("C:/Windows/Fonts/seguisym.ttf", 48)
# from sys import exit
# while True:
#     for event in pygame.event.get():
#         if event.type == pygame.QUIT:
#             pygame.quit()
#             exit()
#     screen.fill('white')
#     render_game_state(env._env.gamestate, screen, font)
#     pygame.display.update()


Yaku counted (with values):
  flowers (combined): 1
  tsumo: 1
  ping_hu: 1
tensor([[[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
         [ 0.0000],
         [ 0.0000],
         [ 0.0000]],

        [[ 0.0000],
  

SystemExit: 

c:\Users\ctc73\PycharmProjects\MahjongAI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


SystemExit: 

In [ ]:
frames_per_batch = 500  # Number of team frames collected per training iteration
n_iters = 5  # Number of sampling and training iterations
total_frames = frames_per_batch * n_iters

collector = Collector(
    env,
    policy,
    device=device,
    storing_device=device,
    frames_per_batch=frames_per_batch,
    total_frames=total_frames
)
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(
        frames_per_batch, device=device
    ),  # We store the frames_per_batch collected at each iteration
    sampler=SamplerWithoutReplacement(),
    batch_size=1,  # We will sample minibatches of this size
)

C:\Users\ctc73\AppData\Local\Temp\ipykernel_34700\3445397793.py:5: FutureWarning: The env passed to Collector is missing transforms required by the policy (InitTracker). From torchrl v0.15 the collector will append them automatically. To enable that behavior now (and silence this warning), pass `auto_register_policy_transforms=True`. To opt out permanently, pass `auto_register_policy_transforms=False`.
  collector = Collector(


In [ ]:
loss_module = ClipPPOLoss(
    actor_network=policy,
    critic_network=critic
)
loss_module.set_keys(  # We have to tell the loss where to find the keys
    reward=env.reward_key,
    action=env.action_key,
    value=("agents", "state_value"),
    # These last 2 keys will be expanded to match the reward shape
    done=("agents", "terminated"),                # per-agent
    terminated=("agents", "terminated"),
)
gamma = 0.995  # discount factor
lmbda = 0.9  # lambda for generalised advantage estimation
lr = 3e-4
loss_module.make_value_estimator(
    ValueEstimators.GAE, gamma=gamma, lmbda=lmbda
)  
GAE = loss_module.value_estimator

optim = torch.optim.Adam(loss_module.parameters(), lr)

In [ ]:
data = env.rollout(10)


100%|██████████| 5/5 [00:00<00:00, 2501.08it/s]


AttributeError: 'TensorDict' object has no attribute 'backward'

In [ ]:
print("\nStarting training...\n")

pbar = tqdm(total=n_iters, desc="Iteration")
episode_reward_mean_list = []

for iter_idx, tensordict_data in enumerate(collector):
    # ---------- Expand terminated to match reward shape ----------
    tensordict_data.set(
        ("next", "agents", "terminated"),
        tensordict_data.get(("next", "terminated"))
        .unsqueeze(-1)
        .expand(tensordict_data.get_item_shape(("next", env.reward_key))),
    )

    # ---------- Compute GAE ----------
    with torch.no_grad():
        GAE(
            tensordict_data,
            params=loss_module.critic_network_params,
            target_params=loss_module.target_critic_network_params,
        )

    # ---------- Store in replay buffer ----------
    data_view = tensordict_data.reshape(-1)
    replay_buffer.extend(data_view)

    # ---------- Mini-batch updates ----------
    loss_vals_avg = {"loss_objective": 0.0, "loss_critic": 0.0, "loss_entropy": 0.0}
    num_batches = frames_per_batch // 1  # batch_size=1, so this is frames_per_batch

    for epoch in range(5):
        for batch_idx in range(num_batches):
            subdata = replay_buffer.sample()
            loss_vals = loss_module(subdata)

            loss_value = (
                loss_vals["loss_objective"]
                + loss_vals["loss_critic"]
                + loss_vals["loss_entropy"]
            )

            loss_value.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), 1.0)
            optim.step()
            optim.zero_grad()

            # Accumulate losses for logging
            for key in loss_vals_avg:
                loss_vals_avg[key] += loss_vals[key].item()

    # Average losses over all batches
    for key in loss_vals_avg:
        loss_vals_avg[key] /= (5 * num_batches)

    collector.update_policy_weights_()

    # ---------- Manual Episode Reward Computation ----------
    rewards = tensordict_data.get(("next", env.reward_key))   # shape [T, 4, 1]
    done_global = tensordict_data.get(("next", "done"))       # global done
    if done_global.dim() > 1:
        done_global = done_global.squeeze(-1)

    done_indices = torch.where(done_global)[0].tolist()
    if len(done_indices) == 0:
        episode_reward_mean = 0.0
    else:
        # Team reward: sum over agents for each timestep
        team_rewards = rewards.sum(dim=1).squeeze(-1)          # [T]
        prev = 0
        episode_returns = []
        for idx in done_indices:
            # Episode from prev to idx (inclusive)
            ep_rew = team_rewards[prev:idx+1].sum().item()
            episode_returns.append(ep_rew)
            prev = idx + 1
        episode_reward_mean = sum(episode_returns) / len(episode_returns)

    episode_reward_mean_list.append(episode_reward_mean)

    # ---------- Logging ----------
    # Compute gradient norm for monitoring
    total_norm = 0.0
    for p in loss_module.parameters():
        if p.grad is not None:
            total_norm += p.grad.norm().item() ** 2
    total_norm = total_norm ** 0.5

    # Advantage stats
    advantage = tensordict_data.get("advantage")
    adv_mean = advantage.mean().item()
    adv_std = advantage.std().item()

    print(f"\nIteration {iter_idx+1}/{n_iters}")
    print(f"  Episode reward mean: {episode_reward_mean:.4f}")
    print(f"  Loss objective: {loss_vals_avg['loss_objective']:.6f}")
    print(f"  Loss critic:    {loss_vals_avg['loss_critic']:.6f}")
    print(f"  Loss entropy:   {loss_vals_avg['loss_entropy']:.6f}")
    print(f"  Advantage mean: {adv_mean:.4f}, std: {adv_std:.4f}")
    print(f"  Gradient norm:  {total_norm:.6f}")
    print("-" * 60)

    pbar.update(1)

print("\nTraining completed!")
print("Episode reward means over iterations:", episode_reward_mean_list)